# CNN training HDF5 demo

This notebook is a lightweight companion to the script-based CNN training workflow.

It does **not** train a full model. The actual training should be launched from the command line with:

```bash
python scripts/train_cnn_hdf5.py --config configs/train_simpleCNN_baseline.json
```

The goal of this notebook is to inspect the training configuration, verify that the HDF5 dataset and split/stat files are consistent, and check one training batch before launching a long run.


## 1. Setup

Run this notebook from the `cbc_pe/` project directory, or adjust `PROJECT_ROOT` below.


In [ ]:
from pathlib import Path
import json
import sys
import os

import h5py
import numpy as np
import torch
from torch.utils.data import DataLoader

# Adjust if needed.
PROJECT_ROOT = Path.cwd()

# If the notebook is run from the repository root instead of cbc_pe/, move into cbc_pe.
if PROJECT_ROOT.name != "cbc_pe" and (PROJECT_ROOT / "cbc_pe").exists():
    PROJECT_ROOT = PROJECT_ROOT / "cbc_pe"

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", Path.cwd())
print("python:", sys.executable)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("cuda version:", torch.version.cuda)
    print("n GPUs:", torch.cuda.device_count())
    print("GPU 0:", torch.cuda.get_device_name(0))


## 2. Load training config

Choose the training config you want to inspect.


In [ ]:
CONFIG_PATH = PROJECT_ROOT / "configs" / "train_simpleCNN_baseline.json"
# CONFIG_PATH = PROJECT_ROOT / "configs" / "train_simpleCNN_pool.json"

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    cfg = json.load(f)

print("Config path:", CONFIG_PATH)
print(json.dumps(cfg, indent=2))


## 3. Resolve input paths

The config contains `project_root` and `data_root`. Those paths may be machine-specific, so this cell resolves the actual HDF5 dataset, split file, and label statistics file used by the training script.


In [ ]:
def resolve_path(base: Path, value: str | None) -> Path | None:
    if value is None:
        return None

    path = Path(value)

    if path.is_absolute():
        return path

    return base / path


project_root_cfg = Path(cfg["project_root"])
data_root = Path(cfg["data_root"])

dataset_cfg = cfg["dataset"]
training_cfg = cfg["training"]

data_processed = data_root / "processed"

dataset_path = resolve_path(data_processed, dataset_cfg["dataset_file"])
split_path = resolve_path(data_processed, dataset_cfg["split_file"])
label_stats_path = resolve_path(data_processed, dataset_cfg["label_stats_file"])

print("project_root in config:", project_root_cfg)
print("data_root:", data_root)
print()
print("dataset_path:", dataset_path)
print("split_path:", split_path)
print("label_stats_path:", label_stats_path)
print()
print("dataset exists:", dataset_path.exists())
print("split exists:", split_path.exists())
print("label stats exists:", label_stats_path.exists())


## 4. Inspect HDF5 dataset

This checks the HDF5 structure and the main metadata attributes.


In [ ]:
with h5py.File(dataset_path, "r") as f:
    print("HDF5 keys:", list(f.keys()))
    print()

    if "X" not in f:
        raise KeyError("HDF5 file does not contain dataset 'X'.")
    if "y" not in f:
        raise KeyError("HDF5 file does not contain dataset 'y'.")

    print("X shape:", f["X"].shape)
    print("X dtype:", f["X"].dtype)
    print("y shape:", f["y"].shape)
    print("y dtype:", f["y"].dtype)

    print()
    print("Important attrs:")
    for key in [
        "num_samples",
        "num_written",
        "dataset_status",
        "detector_names",
        "label_names",
        "duration",
        "length",
        "sampling_frequency",
        "low_frequency_cutoff",
        "processing_length",
    ]:
        if key in f.attrs:
            print(f"{key}: {f.attrs[key]}")

    n_samples = int(f["X"].shape[0])
    num_written = int(f.attrs.get("num_written", n_samples))
    dataset_status = f.attrs.get("dataset_status", "unknown")

    if num_written != n_samples:
        raise ValueError(f"Incomplete dataset: num_written={num_written}, n_samples={n_samples}")
    if dataset_status != "complete":
        raise ValueError(f"Dataset status is not complete: {dataset_status}")


## 5. Load splits and label statistics

The label mean/std must be computed from the train split only. This is what `create_hdf5_splits.py` does.


In [ ]:
splits = np.load(split_path)
stats = np.load(label_stats_path)

print("Split file keys:", splits.files)
print("Stats file keys:", stats.files)
print()

train_idx = splits["train_idx"].astype(np.int64)
val_idx = splits["val_idx"].astype(np.int64)
cal_idx = splits["cal_idx"].astype(np.int64) if "cal_idx" in splits.files else None
test_idx = splits["test_idx"].astype(np.int64) if "test_idx" in splits.files else None

y_mean = stats["y_mean"].astype(np.float32)
y_std = stats["y_std"].astype(np.float32)

label_names = [str(x) for x in stats["label_names"].tolist()] if "label_names" in stats.files else [
    "chirp_mass",
    "total_mass",
    "chi_eff",
]

print("train size:", len(train_idx))
print("val size:", len(val_idx))
print("cal size:", 0 if cal_idx is None else len(cal_idx))
print("test size:", 0 if test_idx is None else len(test_idx))
print()
print("label_names:", label_names)
print("y_mean:", y_mean)
print("y_std:", y_std)

if np.any(y_std <= 0):
    raise ValueError(f"Invalid y_std values: {y_std}")


## 6. Check split overlap

There should be no overlap between train, validation, calibration, and test indices.


In [ ]:
split_dict = {
    "train": train_idx,
    "val": val_idx,
}

if cal_idx is not None:
    split_dict["cal"] = cal_idx

if test_idx is not None:
    split_dict["test"] = test_idx

split_names = list(split_dict.keys())

for i, name_a in enumerate(split_names):
    for name_b in split_names[i + 1:]:
        overlap = np.intersect1d(split_dict[name_a], split_dict[name_b])
        print(f"{name_a} ∩ {name_b}: {len(overlap)}")
        if len(overlap) > 0:
            raise ValueError(f"Overlap detected between {name_a} and {name_b}.")


## 7. Build a small DataLoader

This reproduces the dataset-loading logic used by `train_cnn_hdf5.py`, but only inspects a single batch.


In [ ]:
from src.models.dataset import HDF5RegressionDataset

batch_size = int(training_cfg.get("batch_size", 64))
num_workers = int(training_cfg.get("num_workers", 0))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

train_dataset = HDF5RegressionDataset(
    h5_path=dataset_path,
    indices=train_idx,
    y_mean=y_mean,
    y_std=y_std,
)

val_dataset = HDF5RegressionDataset(
    h5_path=dataset_path,
    indices=val_idx,
    y_mean=y_mean,
    y_std=y_std,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
)

X_batch, y_batch = next(iter(train_loader))

print("device:", device)
print("batch_size:", batch_size)
print("num_workers:", num_workers)
print("pin_memory:", pin_memory)
print()
print("X_batch shape:", X_batch.shape)
print("X_batch dtype:", X_batch.dtype)
print("y_batch shape:", y_batch.shape)
print("y_batch dtype:", y_batch.dtype)
print()
print("X finite:", torch.isfinite(X_batch).all().item())
print("y finite:", torch.isfinite(y_batch).all().item())
print()
print("y batch mean:", y_batch.mean(dim=0))
print("y batch std:", y_batch.std(dim=0))


## 8. Optional: inspect model construction

This checks that the model class named in the config can be imported and built with dimensions inferred from the HDF5 file.


In [ ]:
import importlib

model_cfg = cfg["model"]
class_name = model_cfg["class_name"]
model_kwargs = dict(model_cfg.get("kwargs", {}))

with h5py.File(dataset_path, "r") as f:
    n_detectors = int(f["X"].shape[1])
    n_outputs = int(f["y"].shape[1])
    signal_length = int(f["X"].shape[2])

model_kwargs.setdefault("n_detectors", n_detectors)
model_kwargs.setdefault("n_outputs", n_outputs)

network_module = importlib.import_module("src.models.network")

if not hasattr(network_module, class_name):
    available = [name for name in dir(network_module) if not name.startswith("_")]
    raise AttributeError(f"Model class '{class_name}' not found. Available names include: {available}")

model_class = getattr(network_module, class_name)
model = model_class(**model_kwargs)

print("class_name:", class_name)
print("model_kwargs:", model_kwargs)
print("n_detectors:", n_detectors)
print("signal_length:", signal_length)
print("n_outputs:", n_outputs)
print()
print(model)


## 9. Training command

If all checks above pass, launch training from the terminal, not from this notebook.


In [ ]:
print("Run this from the repository root:")
print()
print(f"python cbc_pe/scripts/train_cnn_hdf5.py --config cbc_pe/configs/{CONFIG_PATH.name}")


## 10. Optional: inspect saved training history

After training, `train_cnn_hdf5.py` saves a compressed `history.npz` file under the configured results directory.

Adjust `HISTORY_PATH` below if you want to plot an existing run.


In [ ]:
import matplotlib.pyplot as plt

# Example only. Update this path to an existing history file.
HISTORY_PATH = None

if HISTORY_PATH is None:
    print("Set HISTORY_PATH to an existing *_history.npz file to plot training curves.")
else:
    history = np.load(HISTORY_PATH)
    print("History keys:", history.files)

    # Common possible keys. Adjust depending on src.models.train.train_model output.
    train_keys = [k for k in history.files if "train" in k.lower() and "loss" in k.lower()]
    val_keys = [k for k in history.files if "val" in k.lower() and "loss" in k.lower()]

    print("train loss keys:", train_keys)
    print("val loss keys:", val_keys)

    if train_keys and val_keys:
        train_loss = history[train_keys[0]]
        val_loss = history[val_keys[0]]

        plt.figure(figsize=(7, 4))
        plt.plot(train_loss, label=train_keys[0])
        plt.plot(val_loss, label=val_keys[0])
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True)
        plt.show()
    else:
        print("Could not infer train/val loss keys automatically.")


## 11. Cleanup

Close HDF5 dataset handles opened by `HDF5RegressionDataset`.


In [ ]:
train_dataset.close()
val_dataset.close()

print("Closed HDF5 dataset handles.")
